# YOLOE Stage Explorer

Use this notebook to inspect YOLOE detection results produced by the pipeline. You'll be able to:
- review detection counts by class and attribute
- slice detections with dropdown filters
- inspect sample rows before jumping back into the full viewer


## Quick start

1. Activate the same Python environment you use for the privacy pipeline.
2. Run each cell from top to bottom, select the YOLOE output (base run or a filtered run), and start exploring.


In [69]:
import json
from pathlib import Path

import yaml

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install pandas to use this notebook (pip install pandas)."
    ) from exc

try:
    import plotly.express as px
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install plotly to use this notebook (pip install plotly)."
    ) from exc

try:
    import ipywidgets as widgets
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install ipywidgets to use this notebook (pip install ipywidgets)."
    ) from exc

from IPython.display import Markdown, display, Image as IPImage, Image as IPImage, Image as IPImage

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_colwidth", 120)
px.defaults.template = "plotly_white"


In [70]:
def find_project_root(markers=("pyproject.toml", ".git")):
    start = Path.cwd()
    for candidate in [start, *start.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return start

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

CUSTOM_CONFIG_PATH = None  # Override with Path("/custom/config/config.yaml") if needed
config_path = Path(CUSTOM_CONFIG_PATH) if CUSTOM_CONFIG_PATH else PROJECT_ROOT / "config/config.yaml"
if not config_path.exists():
    raise FileNotFoundError(
        f"Could not find a config file next to this notebook. Expected {config_path}."
    )
print(f"Loading config from: {config_path}")

with config_path.open() as f:
    config_data = yaml.safe_load(f) or {}


def resolve_path(path):
    if path is None:
        return None
    candidate = Path(path)
    if not candidate.is_absolute():
        candidate = PROJECT_ROOT / candidate
    return candidate


def load_jsonl(path):
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def discover_outputs(stage_name, filenames):
    outputs = {}
    seen = set()
    ordered_names = [str(name) for name in filenames if name]
    for name in ordered_names:
        candidate = resolve_path(name)
        if candidate and candidate.exists():
            label = f"default run: {name}"
            outputs[label] = candidate
            seen.add(candidate.resolve())
    stage_dir = PROJECT_ROOT / "filtered" / stage_name
    if stage_dir.exists():
        for option in sorted(stage_dir.glob("*")):
            if not option.is_dir():
                continue
            for name in ordered_names:
                candidate = option / name
                if candidate.exists() and candidate.resolve() not in seen:
                    label = f"output/filtered/{stage_name}/{option.name} -> {name}"
                    outputs[label] = candidate
                    seen.add(candidate.resolve())
    return outputs


gemini_cfg = config_data.get("gemini", {}) if isinstance(config_data, dict) else {}
SCENE_DIRECTORY_LEVEL = int((dataset_cfg.get("scene_directory_level") or gemini_cfg.get("scene_directory_level") or 1))
print(f"Scene grouping level: {SCENE_DIRECTORY_LEVEL}")


Project root: /Users/bchauhan/Downloads/data-privacy-tool
Loading config from: /Users/bchauhan/Downloads/data-privacy-tool/config/config.yaml
Scene grouping level: 2


In [71]:
yolo_cfg = config_data.get("yoloe", {}) if isinstance(config_data, dict) else {}
default_output_name = yolo_cfg.get("output_jsonl", "yoloe_output.jsonl")
candidate_names = {default_output_name, "yoloe_output.jsonl"}
yolo_sources = discover_outputs("yoloe", candidate_names)

if not yolo_sources:
    raise FileNotFoundError(
        "No YOLOE output JSONL files were found. Run the pipeline first or update candidate_names."
    )

yolo_source_picker = widgets.Dropdown(
    options=[(label, str(path)) for label, path in yolo_sources.items()],
    description="YOLOE output",
    layout=widgets.Layout(width="80%"),
)

display(Markdown("**Select the YOLOE result set you want to explore:**"))
display(yolo_source_picker)


**Select the YOLOE result set you want to explore:**

Dropdown(description='YOLOE output', layout=Layout(width='80%'), options=(('default run: output/yoloe_output.j…

In [72]:

selected_yolo_path = Path(yolo_source_picker.value)
yolo_records = load_jsonl(selected_yolo_path)

if not yolo_records:
    raise ValueError(f"No records found in {selected_yolo_path}")

scene_levels_up = max(1, int(SCENE_DIRECTORY_LEVEL))

def derive_scene_path(image_path_str):
    if not image_path_str:
        return None
    current = Path(image_path_str)
    for _ in range(scene_levels_up):
        current = current.parent
    return str(current)

images_df = pd.DataFrame(yolo_records)
if "image_path" not in images_df.columns:
    images_df["image_path"] = None
if "attributes" in images_df.columns:
    attributes_series = images_df["attributes"].apply(lambda x: x or {})
else:
    attributes_series = pd.Series([{} for _ in range(len(images_df))])
attributes_df = pd.json_normalize(attributes_series)
attribute_columns = attributes_df.columns.tolist()
for column in attribute_columns:
    images_df[column] = attributes_df[column]

images_df["scene_path"] = images_df["image_path"].apply(derive_scene_path)
if "scene_path" not in attribute_columns:
    attribute_columns.append("scene_path")

if "detections" not in images_df.columns:
    images_df["detections"] = [[] for _ in range(len(images_df))]
images_df["detections"] = images_df["detections"].apply(lambda x: x or [])
images_df["num_detections"] = images_df["detections"].apply(len)
images_df["has_detection"] = images_df["num_detections"] > 0


images_df["detected_classes"] = images_df["detections"].apply(
    lambda dets: [d.get("class") for d in dets if isinstance(d, dict) and d.get("class")]
)

path_columns = [col for col in images_df.columns if "path" in col.lower()]
ordered_columns: list[str] = []
if "detected_classes" in images_df.columns:
    ordered_columns.append("detected_classes")
for attr in attribute_columns:
    if attr in path_columns:
        continue
    if attr in images_df.columns and attr not in ordered_columns:
        ordered_columns.append(attr)
non_path_columns = [col for col in images_df.columns if col not in set(path_columns + ordered_columns)]
ordered_columns.extend(non_path_columns)
ordered_columns.extend([col for col in path_columns if col not in ordered_columns])
images_df = images_df.loc[:, ordered_columns]



def _row_attributes(record_attributes):
    if not record_attributes:
        return {}
    return {key: value for key, value in record_attributes.items()}


detection_rows = []
for record in yolo_records:
    attributes = _row_attributes(record.get("attributes"))
    detections = record.get("detections", []) or []
    scene_path = derive_scene_path(record.get("image_path"))
    for detection in detections:
        entry = {
            "image_path": record.get("image_path"),
            "display_path": record.get("visualization_path") or record.get("image_path"),
            "class": detection.get("class"),
            "confidence": detection.get("confidence", 0.0),
            "bbox": detection.get("bbox"),
            "scene_path": scene_path,
        }
        for key, value in attributes.items():
            entry[key] = value
        detection_rows.append(entry)


detections_df = pd.DataFrame(detection_rows)
class_names = sorted(detections_df["class"].dropna().unique()) if not detections_df.empty else []


## Provide visibility into attribute columns even if detections_df is empty
for column in attribute_columns:
    if column not in images_df.columns:
        images_df[column] = None


# Display high-level summary
display(
    Markdown(
        f"Loaded **{len(images_df):,}** images and **{len(detections_df):,}** detections from `{selected_yolo_path}`."
    )
)
if attribute_columns:
    display(Markdown(f"Available attributes: {', '.join(attribute_columns)}"))
else:
    display(Markdown("This run did not record any custom attributes."))

if detections_df.empty:
    display(
        Markdown(
            "⚠️ No detections were found in this file; the widgets below will stay empty until detections are available."
        )
    )

images_df.head()


Loaded **22** images and **26** detections from `/Users/bchauhan/Downloads/data-privacy-tool/output/yoloe_output.jsonl`.

Available attributes: building, scene, date, scene_path

,detected_classes,building,scene,date,attributes,detections,num_detections,has_detection,image_path,visualization_path,scene_path
0,[person],Smith_Hall_121,500180237,Mon_Jul_10_11:45:07_2023,"{'building': 'Smith_Hall_121', 'scene': '500180237', 'date': 'Mon_Jul_10_11:45:07_2023'}","[{'class': 'person', 'confidence': 0.5193725228309631, 'bbox': [552.42333984375, 0.00250244140625, 1279.4775390625, ...",1,True,data/Smith_Hall_121/500180237/Mon_Jul_10_11:45:07_2023/16291792/000022.png,yoloe_visualizations/Smith_Hall_121/500180237/Mon_Jul_10_11:45:07_2023/16291792/000022.png,data/Smith_Hall_121/500180237/Mon_Jul_10_11:45:07_2023
1,[whiteboard],Smith_Hall_121,500180237,Mon_Jul_10_11:29:04_2023,"{'building': 'Smith_Hall_121', 'scene': '500180237', 'date': 'Mon_Jul_10_11:29:04_2023'}","[{'class': 'whiteboard', 'confidence': 0.6943395137786865, 'bbox': [0.0, 0.0, 293.19586181640625, 301.8177490234375]}]",1,True,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000001.png,yoloe_visualizations/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000001.png,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023
2,[whiteboard],Smith_Hall_121,500180237,Mon_Jul_10_11:29:04_2023,"{'building': 'Smith_Hall_121', 'scene': '500180237', 'date': 'Mon_Jul_10_11:29:04_2023'}","[{'class': 'whiteboard', 'confidence': 0.7566551566123962, 'bbox': [0.0, 0.27252197265625, 293.2254333496094, 300.05...",1,True,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000002.png,yoloe_visualizations/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000002.png,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023
3,"[poster, whiteboard]",Smith_Hall_121,500180237,Mon_Jul_10_11:29:04_2023,"{'building': 'Smith_Hall_121', 'scene': '500180237', 'date': 'Mon_Jul_10_11:29:04_2023'}","[{'class': 'poster', 'confidence': 0.6288219690322876, 'bbox': [0.0, 0.1932373046875, 293.729736328125, 295.38287353...",2,True,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000003.png,yoloe_visualizations/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023/26638268/000003.png,data/Smith_Hall_121/500180237/Mon_Jul_10_11:29:04_2023
4,"[poster, whiteboard]",Smith_Hall_121,500180237,Mon_Jul_10_10:52:08_2023,"{'building': 'Smith_Hall_121', 'scene': '500180237', 'date': 'Mon_Jul_10_10:52:08_2023'}","[{'class': 'poster', 'confidence': 0.6918227672576904, 'bbox': [0.0, 0.13433837890625, 289.0703125, 295.014770507812...",2,True,data/Smith_Hall_121/500180237/Mon_Jul_10_10:52:08_2023/26638268/000001.png,yoloe_visualizations/Smith_Hall_121/500180237/Mon_Jul_10_10:52:08_2023/26638268/000001.png,data/Smith_Hall_121/500180237/Mon_Jul_10_10:52:08_2023


## Filter and visualize detections

Use the controls below to slice detections by attribute, class, and confidence. Every visualization listens to the same filters.


In [73]:
attribute_widgets = {}
for attr in attribute_columns:
    values = sorted({str(v) for v in images_df[attr].dropna().unique()})
    options = ["All"] + values
    attribute_widgets[attr] = widgets.Dropdown(
        options=options,
        value="All",
        description=attr.replace("_", " ").title(),
        layout=widgets.Layout(width="280px"),
    )

class_widget = widgets.SelectMultiple(
    options=class_names,
    value=tuple(class_names) if class_names else (),
    description="Classes",
    rows=min(max(len(class_names), 3), 10) if class_names else 3,
    layout=widgets.Layout(width="320px"),
)

confidence_slider = widgets.FloatSlider(
    min=0.0,
    max=1.0,
    step=0.05,
    value=0.5,
    description="Min confidence",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="50%"),
)

rows_slider = widgets.IntSlider(
    min=5,
    max=40,
    step=5,
    value=10,
    description="Rows to preview",
    continuous_update=False,
)

if attribute_columns:
    group_attr_widget = widgets.Dropdown(
        options=attribute_columns,
        value=attribute_columns[0],
        description="Group attribute",
        layout=widgets.Layout(width="50%"),
    )
else:
    group_attr_widget = None

controls_children = [widgets.HTML("<b>Filter detections</b>")]
if attribute_widgets:
    attr_widgets = list(attribute_widgets.values())
    for start in range(0, len(attr_widgets), 2):
        controls_children.append(widgets.HBox(attr_widgets[start : start + 2]))
controls_children.extend([widgets.HBox([class_widget]), confidence_slider])
display(widgets.VBox(controls_children))

shared_controls = {f"attr__{attr}": widget for attr, widget in attribute_widgets.items()}
shared_controls["class_filter"] = class_widget
shared_controls["min_confidence"] = confidence_slider


def _filter_detections_from_kwargs(kwargs):
    data = detections_df.copy()
    if data.empty:
        return data
    for attr in attribute_columns:
        selection = kwargs.get(f"attr__{attr}")
        if selection and selection != "All":
            data = data[data[attr].astype(str) == str(selection)]
    class_selection = kwargs.get("class_filter") or []
    if class_selection:
        data = data[data["class"].isin(class_selection)]
    min_conf = kwargs.get("min_confidence", 0.0)
    data = data[data["confidence"].fillna(0) >= float(min_conf)]
    return data


def render_selection_summary(**kwargs):
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No detections match the current filters."))
        return
    summary = {
        "Images in selection": int(filtered["image_path"].nunique()),
        "Detections in selection": int(len(filtered)),
        "Unique classes": int(filtered["class"].nunique()),
    }
    total = len(detections_df)
    if total:
        summary["Share of all detections"] = f"{len(filtered) / total * 100:.1f}%"
    display(pd.DataFrame([summary]))


def render_class_distribution(**kwargs):
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No detections to chart."))
        return
    class_counts = (
        filtered.groupby("class")
        .size()
        .reset_index(name="detections")
        .sort_values("detections", ascending=False)
    )
    fig = px.bar(class_counts, x="class", y="detections", text_auto=".0f", title="Detections per class")
    fig.update_layout(xaxis_title="Class", yaxis_title="Detections")
    fig.show()


def render_attribute_stack(group_attribute=None, **kwargs):
    if not group_attribute:
        display(Markdown("Choose an attribute to build the stacked chart."))
        return
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No detections to chart."))
        return
    if group_attribute not in filtered.columns:
        display(Markdown(f"{group_attribute} is not present in this data."))
        return
    grouped = (
        filtered.groupby([group_attribute, "class"])
        .size()
        .reset_index(name="detections")
    )
    grouped = grouped[grouped[group_attribute].notna()]
    if grouped.empty:
        display(Markdown("No rows left after grouping."))
        return
    fig = px.bar(
        grouped,
        x=group_attribute,
        y="detections",
        color="class",
        text_auto=".0f",
        title=f"Detections by {group_attribute} and class",
    )
    fig.update_layout(barmode="stack", xaxis_title=group_attribute, yaxis_title="Detections")
    fig.show()


def render_confidence_histogram(**kwargs):
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No detections to chart."))
        return
    fig = px.histogram(
        filtered,
        x="confidence",
        color="class",
        nbins=20,
        opacity=0.8,
        title="Confidence distribution",
    )
    fig.update_layout(xaxis_title="Confidence", yaxis_title="Count")
    fig.show()


def render_sample_table(rows_to_show=10, **kwargs):
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        display(Markdown("No detections to preview."))
        return
    columns = ["image_path", "class", "confidence", "bbox", "display_path"] + attribute_columns
    available_cols = [col for col in columns if col in filtered.columns]
    preview = filtered[available_cols].copy()
    preview["confidence"] = preview["confidence"].map(lambda x: round(float(x), 3) if pd.notna(x) else x)
    display(preview.head(rows_to_show))


summary_panel = widgets.interactive_output(render_selection_summary, dict(shared_controls))
class_panel = widgets.interactive_output(render_class_distribution, dict(shared_controls))
conf_panel = widgets.interactive_output(render_confidence_histogram, dict(shared_controls))
table_controls = dict(shared_controls)
table_controls["rows_to_show"] = rows_slider
table_panel = widgets.interactive_output(render_sample_table, table_controls)


display(Markdown("### Selection summary"))
display(summary_panel)


display(Markdown("### Detections by class"))
display(class_panel)

if group_attr_widget:
    group_controls = dict(shared_controls)
    group_controls["group_attribute"] = group_attr_widget
    stacked_panel = widgets.interactive_output(render_attribute_stack, group_controls)
    display(Markdown("### Attribute + class breakdown"))
    display(group_attr_widget)
    display(stacked_panel)


display(Markdown("### Confidence distribution"))
display(conf_panel)



### Selection summary

Output()

### Detections by class

Output()

### Attribute + class breakdown

Dropdown(description='Group attribute', layout=Layout(width='50%'), options=('building', 'scene', 'date', 'sce…

Output()

### Confidence distribution

Output()


### Scene gallery

Scenes are grouped using the dataset `scene_directory_level` so each page represents a single scene. Adjust the mirrored filters below (they stay in sync with the controls above), pick how many images per scene to preview, and page through the detections while viewing every image that matched your criteria.


In [74]:

import math

gallery_state = {"scenes": [], "page": 1, "suspend_page_events": False}
gallery_status = widgets.HTML("<em>Adjust the gallery filters to load scenes.</em>")
gallery_prev_button = widgets.Button(description="Previous", icon="arrow-left", disabled=True)
gallery_next_button = widgets.Button(description="Next", icon="arrow-right", disabled=True)
gallery_page_dropdown = widgets.Dropdown(
    options=[("Scene 1 / 1", 1)],
    value=1,
    description="Scene",
    disabled=True,
    layout=widgets.Layout(width="240px"),
)
gallery_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="0.75em"))

# Mirror the primary filters so they can be adjusted near the gallery as well
linked_controls = []
gallery_attribute_clones = {}
for attr, widget in attribute_widgets.items():
    clone = widgets.Dropdown(
        options=list(widget.options),
        value=widget.value,
        description=widget.description,
        layout=widgets.Layout(width="280px"),
    )
    linked_controls.append(widgets.link((clone, "value"), (widget, "value")))
    gallery_attribute_clones[attr] = clone

gallery_class_clone = widgets.SelectMultiple(
    options=list(class_widget.options),
    value=tuple(class_widget.value) if isinstance(class_widget.value, tuple) else class_widget.value,
    description=class_widget.description,
    rows=class_widget.rows,
    layout=widgets.Layout(width="320px"),
)
linked_controls.append(widgets.link((gallery_class_clone, "value"), (class_widget, "value")))

gallery_conf_clone = widgets.FloatSlider(
    min=confidence_slider.min,
    max=confidence_slider.max,
    step=confidence_slider.step,
    value=confidence_slider.value,
    description=confidence_slider.description,
    readout_format=confidence_slider.readout_format,
    continuous_update=False,
    layout=widgets.Layout(width="50%"),
)
linked_controls.append(widgets.link((gallery_conf_clone, "value"), (confidence_slider, "value")))

gallery_controls_children = [widgets.HTML("<b>Scene filters (mirrored from above)</b>")]
if gallery_attribute_clones:
    attr_widgets_clone = list(gallery_attribute_clones.values())
    for start in range(0, len(attr_widgets_clone), 2):
        gallery_controls_children.append(widgets.HBox(attr_widgets_clone[start : start + 2]))
gallery_controls_children.extend([widgets.HBox([gallery_class_clone]), gallery_conf_clone])
display(widgets.VBox(gallery_controls_children))

def _records_for_gallery(kwargs):
    filtered = _filter_detections_from_kwargs(kwargs)
    if filtered.empty:
        return []
    working = filtered.copy()
    if "scene_path" not in working.columns:
        working["scene_path"] = working["image_path"].apply(derive_scene_path)
    grouped_scenes = []
    for scene_path, group in working.groupby("scene_path", dropna=False):
        ordered = group.sort_values(["image_path", "confidence"], ascending=[True, False])
        scene_attributes = {}
        for attr in attribute_columns:
            if attr == "scene_path" or attr not in ordered.columns:
                continue
            series = ordered[attr].dropna()
            if not series.empty:
                scene_attributes[attr] = series.iloc[0]
        images_map = {}
        images = []
        for _, row in ordered.iterrows():
            image_path = row.get("image_path")
            display_path = row.get("display_path") or image_path
            image_key = display_path or image_path
            if not image_key:
                continue
            if image_key not in images_map:
                image_attributes = {attr: row.get(attr) for attr in attribute_columns if attr in ordered.columns}
                images_map[image_key] = {
                    "image_path": image_path,
                    "display_path": display_path,
                    "attributes": image_attributes,
                    "detections": [],
                }
                images.append(images_map[image_key])
            detection_entry = {
                "class": row.get("class"),
                "confidence": row.get("confidence"),
                "bbox": row.get("bbox"),
            }
            images_map[image_key]["detections"].append(detection_entry)
        grouped_scenes.append(
            {
                "scene_path": scene_path,
                "attributes": scene_attributes,
                "images": images,
            }
        )
    grouped_scenes.sort(key=lambda entry: entry["scene_path"] or "")
    return grouped_scenes



def _render_gallery():
    gallery_output.clear_output()
    total = len(gallery_state["scenes"])
    gallery_state["suspend_page_events"] = True
    try:
        if total == 0:
            gallery_status.value = "<em>No scenes match the current filters.</em>"
            gallery_page_dropdown.options = [("Scene 1 / 1", 1)]
            gallery_page_dropdown.value = 1
            gallery_page_dropdown.disabled = True
            gallery_prev_button.disabled = True
            gallery_next_button.disabled = True
            with gallery_output:
                display(Markdown("No scenes available for the current filters."))
            return

        total_pages = total
        page = min(max(1, gallery_state["page"]), total_pages)
        gallery_state["page"] = page
        gallery_page_dropdown.options = [(f"Scene {i + 1} / {total_pages}", i + 1) for i in range(total_pages)]
        gallery_page_dropdown.value = page
        gallery_page_dropdown.disabled = total_pages <= 1
        gallery_prev_button.disabled = page <= 1
        gallery_next_button.disabled = page >= total_pages

        scene_entry = gallery_state["scenes"][page - 1]
        scene_label = scene_entry.get("scene_path") or "(unknown scene)"
        raw_images = scene_entry.get("images", [])
        subset = []
        seen_keys = set()
        for entry in raw_images:
            key = entry.get("display_path") or entry.get("image_path")
            if not key or key in seen_keys:
                continue
            seen_keys.add(key)
            subset.append(entry)
        scene_entry["images"] = subset
        total_images = len(subset)
        gallery_status.value = f"Scene {page} of {total_pages} — {total_images} images"

        with gallery_output:
            display(Markdown(f"**Scene `{scene_label}`**"))
            scene_attrs = scene_entry.get("attributes") or {}
            attr_bits = []
            for attr, value in scene_attrs.items():
                if pd.notna(value):
                    attr_bits.append(f"{attr}: {value}")
            if attr_bits:
                display(Markdown("*" + ", ".join(attr_bits) + "*"))
            if total_images == 0:
                display(Markdown("No images available for this scene."))
            for idx, entry in enumerate(subset, start=1):
                display_path = entry.get("display_path") or entry.get("image_path")
                detections = entry.get("detections") or []
                if detections:
                    bullets = []
                    for det in detections:
                        cls = det.get("class") or "Unknown"
                        conf = det.get("confidence")
                        if pd.notna(conf):
                            bullets.append(f"- {cls} ({float(conf):.2f})")
                        else:
                            bullets.append(f"- {cls}")
                    summary = "\n".join(bullets)
                else:
                    summary = "_No detections recorded_"
                display(Markdown(f"**Image {idx}** — `{display_path}`"))
                display(Markdown(summary))
                image_attributes = entry.get("attributes") or {}
                meta_bits = []
                for attr, value in image_attributes.items():
                    if attr == "scene_path" or pd.isna(value):
                        continue
                    meta_bits.append(f"{attr}: {value}")
                if meta_bits:
                    display(Markdown("*" + ", ".join(meta_bits) + "*"))
                resolved = _resolve_image_path(display_path)
                if resolved and resolved.exists():
                    display(IPImage(filename=str(resolved), width=360))
                else:
                    display(Markdown("_Image not found locally._"))
                display(Markdown("---"))
    finally:
        gallery_state["suspend_page_events"] = False


def _resolve_image_path(path_str):
    if not path_str:
        return None
    candidate = Path(path_str)
    if not candidate.is_absolute():
        candidate = (PROJECT_ROOT / candidate).resolve()
    return candidate


def _refresh_gallery(**kwargs):
    gallery_state["scenes"] = _records_for_gallery(kwargs)
    gallery_state["page"] = 1
    _render_gallery()


def _on_prev(_):
    current = gallery_page_dropdown.value or 1
    if current > 1:
        gallery_page_dropdown.value = current - 1


def _on_next(_):
    total = len(gallery_state["scenes"])
    current = gallery_page_dropdown.value or 1
    if current < total:
        gallery_page_dropdown.value = current + 1


def _on_page_change(change):
    if gallery_state.get("suspend_page_events"):
        return
    if change.get("name") == "value" and change.get("new"):
        gallery_state["page"] = int(change["new"])
        _render_gallery()



gallery_prev_button.on_click(_on_prev)
gallery_next_button.on_click(_on_next)
gallery_page_dropdown.observe(_on_page_change, names="value")

gallery_filters = dict(shared_controls)
gallery_trigger = widgets.interactive_output(_refresh_gallery, gallery_filters)
gallery_trigger.layout.display = "none"

display(gallery_status)
display(widgets.HBox([gallery_prev_button, gallery_page_dropdown, gallery_next_button]))
display(gallery_output)
display(gallery_trigger)


HTML(value='Scene 1 of 7 — 3 images')

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

Output(layout=Layout(display='none'))